In [ ]:


# imports

import os
import requests
from dotenv import load_dotenv
from openai import OpenAI
from IPython.display import Markdown, display

load_dotenv(override=True)
openai_api_key = os.getenv('OPENAI_API_KEY')
anthropic_api_key = os.getenv('ANTHROPIC_API_KEY')

if openai_api_key:
    print(f"OpenAI API Key exists and begins {openai_api_key[:8]}")
else:
    print("OpenAI API Key not set")
    
if anthropic_api_key:
    print(f"Anthropic API Key exists and begins {anthropic_api_key[:7]}")
else:
    print("Anthropic API Key not set (and this is optional)")

# Connect to OpenAI client library
# A thin wrapper around calls to HTTP endpoints

openai = OpenAI()

# For Gemini, DeepSeek and Groq, we can use the OpenAI python client
# Because Google and DeepSeek have endpoints compatible with OpenAI
# And OpenAI allows you to change the base_url

anthropic_url = "https://api.anthropic.com/v1/"
ollama_url = "http://localhost:11434/v1"

anthropic = OpenAI(api_key=anthropic_api_key, base_url=anthropic_url) if anthropic_api_key else None
ollama = OpenAI(api_key="ollama", base_url=ollama_url)

requests.get("http://localhost:11434/").content


!ollama pull llama3.2




# We will create a conversation between 3 LLMs:
# 1. GPT-4.1-mini
# 2. Claude Haiku 4.5
# 3. Llama 3.2 running locally with Ollama


# -----------------------------------
# MODEL NAMES
# -----------------------------------

gpt_model = "gpt-4.1-mini"
claude_model = "claude-haiku-4-5"
llama_model = "llama3.2"


# -----------------------------------
# PERSONALITY / SYSTEM PROMPTS
# -----------------------------------

# GPT will be argumentative
gpt_system = """You are a chatbot who is very argumentative;
you disagree with anything in the conversation and you challenge everything, in a snarky way."""


# Claude will be polite and try to find common ground
claude_system = """You are a very polite, courteous chatbot. You try to agree with
everything the other person says, or find common ground. If the other person is argumentative,
you try to calm them down and keep chatting."""


# Llama will act as the neutral third participant
llama_system = """You are a neutral chatbot. You listen to both sides of the conversation
and give your own balanced opinion."""


# -----------------------------------
# SHARED CONVERSATION HISTORY
# -----------------------------------

# Instead of having separate lists for GPT and Claude,
# we now keep ONE conversation history for all 3 models.
#
# Every message contains:
# - speaker: who said it
# - content: what they said

conversation = [
    {"speaker": "GPT", "content": "Hi there"},
    {"speaker": "Claude", "content": "Hi"}
]


# -----------------------------------
# BUILD THE MESSAGE HISTORY
# -----------------------------------

def build_messages(system_prompt):

    # Start with the personality/instructions for the current model
    messages = [
        {"role": "system", "content": system_prompt}
    ]

    # Add every previous message from the shared conversation
    for item in conversation:

        # Example:
        # "GPT: I disagree with that"
        # "Claude: I understand your point"
        #
        # We send them as user messages because each model is reading
        # the conversation as text and deciding what to say next.

        messages.append({
            "role": "user",
            "content": f"{item['speaker']}: {item['content']}"
        })

    return messages


# -----------------------------------
# CALL GPT
# -----------------------------------

def call_gpt():

    # Send GPT its personality + the full conversation
    response = openai.chat.completions.create(
        model=gpt_model,
        messages=build_messages(gpt_system)
    )

    # Return only GPT's text response
    return response.choices[0].message.content


# -----------------------------------
# CALL CLAUDE
# -----------------------------------

def call_claude():

    # Same idea, but using the Anthropic-compatible client
    response = anthropic.chat.completions.create(
        model=claude_model,
        messages=build_messages(claude_system)
    )

    return response.choices[0].message.content


# -----------------------------------
# CALL LOCAL LLAMA THROUGH OLLAMA
# -----------------------------------

def call_llama():

    # Ollama exposes an OpenAI-compatible API,
    # so the code looks almost identical to GPT.
    response = ollama.chat.completions.create(
        model=llama_model,
        messages=build_messages(llama_system)
    )

    return response.choices[0].message.content


# -----------------------------------
# DISPLAY THE STARTING MESSAGES
# -----------------------------------

display(Markdown(
    f"### GPT:\n{conversation[0]['content']}\n"
))

display(Markdown(
    f"### Claude:\n{conversation[1]['content']}\n"
))


# -----------------------------------
# RUN THE CONVERSATION
# -----------------------------------

# Run 5 rounds.
#
# Each round is:
#
# GPT -> Claude -> Llama
#
# Every response is added to the shared conversation,
# so the next model can see what the previous models said.

for i in range(5):

    # ----- GPT'S TURN -----

    gpt_next = call_gpt()

    # Save GPT's response in the shared conversation
    conversation.append({
        "speaker": "GPT",
        "content": gpt_next
    })

    # Show GPT's response in the notebook
    display(Markdown(
        f"### GPT:\n{gpt_next}\n"
    ))


    # ----- CLAUDE'S TURN -----

    # Claude now sees GPT's new response because
    # it was just added to `conversation`
    claude_next = call_claude()

    conversation.append({
        "speaker": "Claude",
        "content": claude_next
    })

    display(Markdown(
        f"### Claude:\n{claude_next}\n"
    ))


    # ----- LLAMA'S TURN -----

    # Llama now sees both GPT's and Claude's latest responses
    llama_next = call_llama()

    conversation.append({
        "speaker": "OLlama",
        "content": llama_next
    })

    display(Markdown(
        f"### Llama:\n{llama_next}\n"
    ))